# Caso Empresarial MANACO - Diagnóstico mediante Ciencia de Datos
## Estación 2: Exploración, Indicadores y Diagnóstico Estratégico (30 minutos)

**Programa:** Ingeniería de Sistemas, Ciencia de Datos e Inteligencia de Negocios (6.° Semestre)  
**Modalidad:** Aprendizaje Basado en Retos (ABR)  
**Rol:** Equipo Consultor de Ciencia de Datos para la Gerencia General de Manaco

---

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Localización robusta de la raíz del proyecto y dataset
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

raw_dir = ROOT / "data" / "raw"
archivos_xlsx = list(raw_dir.glob("*.xlsx"))
if not archivos_xlsx:
    raise FileNotFoundError(f"No se encontró ningún archivo Excel en {raw_dir.resolve()}")

archivo = archivos_xlsx[0]
print(f"Cargando archivo: {archivo.name}")
df = pd.read_excel(archivo, sheet_name="Base de Datos")

print(f"Dimensiones del dataset: {df.shape[0]} filas x {df.shape[1]} columnas.")
display(df.head())
print("\n--- Información de Columnas y Tipos ---")
df.info()
print("\n--- Valores Nulos por Campo ---")
display(df.isna().sum().to_frame("Faltantes"))

### 1. Indicadores Globales y Regla del Valor por Par Ponderado

**Regla Fundamental:**  
$$\text{Valor por Par Ponderado} = \frac{\sum \text{Valor Comercial USD}}{\sum \text{Cantidad Pares}}$$
*Evitar el promedio simple de la columna unitaria `Valor_USD_por_par`.*

In [ ]:
resumen_flujo = df.groupby("Flujo_comercial").agg(
    Registros=("Valor_USD", "count"),
    Valor_Total_USD=("Valor_USD", "sum"),
    Cantidad_Total_Pares=("Cantidad_pares", "sum"),
    Promedio_Simple_Unitario=("Valor_USD_por_par", "mean")
).reset_index()

resumen_flujo["Valor_por_Par_Ponderado"] = (resumen_flujo["Valor_Total_USD"] / resumen_flujo["Cantidad_Total_Pares"]).round(2)
resumen_flujo["Valor_Millones_USD"] = (resumen_flujo["Valor_Total_USD"] / 1e6).round(2)
resumen_flujo["Pares_Millones"] = (resumen_flujo["Cantidad_Total_Pares"] / 1e6).round(2)

display(resumen_flujo[["Flujo_comercial", "Registros", "Valor_Millones_USD", "Pares_Millones", "Valor_por_Par_Ponderado", "Promedio_Simple_Unitario"]])

deficit_usd = resumen_flujo.loc[resumen_flujo["Flujo_comercial"]=="Importación", "Valor_Total_USD"].values[0] - \
              resumen_flujo.loc[resumen_flujo["Flujo_comercial"]=="Exportación", "Valor_Total_USD"].values[0]
print(f"Déficit comercial global: ${deficit_usd/1e6:,.2f} Millones de USD a favor de las importaciones.")

### 2. Gráfico 1: Importaciones frente a Exportaciones por Año (2020 - 2024)
Evolución comparativa del comercio exterior registrado.

In [ ]:
plt.figure(figsize=(10, 5.5))
sns.set_theme(style="whitegrid", font_scale=1.1)

anual = df.groupby(["Año", "Flujo_comercial"])["Valor_USD"].sum().reset_index()
anual["Valor_Millones"] = anual["Valor_USD"] / 1e6

sns.barplot(data=anual, x="Año", y="Valor_Millones", hue="Flujo_comercial", palette=["#1f77b4", "#d62728"])
plt.title("Gráfico 1: Importaciones frente a Exportaciones por Año (2020 - 2024)", fontsize=14, fontweight="bold", pad=12)
plt.xlabel("Año", labelpad=8)
plt.ylabel("Valor Comercial (Millones de USD)", labelpad=8)
plt.legend(title="Flujo Comercial", frameon=True)
plt.tight_layout()
plt.show()

### 3. Gráfico 2: Principales Socios Comerciales por Valor de Importación
Identificación de la concentración geográfica en proveedores.

In [ ]:
plt.figure(figsize=(10, 6))
top_socios = df[df["Flujo_comercial"] == "Importación"].groupby("Socio_comercial")["Valor_USD"].sum().reset_index()
top_socios["Valor_Millones"] = top_socios["Valor_USD"] / 1e6
top_socios = top_socios.sort_values(by="Valor_Millones", ascending=False).head(10)

sns.barplot(data=top_socios, x="Valor_Millones", y="Socio_comercial", hue="Socio_comercial", palette="Blues_r", legend=False)
plt.title("Gráfico 2: Top 10 Socios Comerciales por Valor de Importación (Millones USD)", fontsize=14, fontweight="bold", pad=12)
plt.xlabel("Valor Importado (Millones de USD)", labelpad=8)
plt.ylabel("Socio Comercial", labelpad=8)
plt.tight_layout()
plt.show()

### 4. Gráfico 3: Comparación de Valor por Par por Categoría de Calzado
Diferenciación entre el calzado sintético/deportivo y el calzado de cuero.

In [ ]:
cat = df.groupby(["Categoría_calzado", "Flujo_comercial"]).agg(
    Valor_Total=("Valor_USD", "sum"),
    Pares_Total=("Cantidad_pares", "sum")
).reset_index()
cat["Valor_por_Par"] = cat["Valor_Total"] / cat["Pares_Total"]
cat["Categoria_Corta"] = cat["Categoría_calzado"].map({
    "Calzado deportivo con suela de caucho/plástico": "Deportivo (Caucho/Plástico)",
    "Calzado con suela de caucho/plástico y parte superior de cuero": "Suela Sintética / Cuero",
    "Calzado con suela de cuero y parte superior de cuero/material similar": "Suela Cuero / Cuero"
})

plt.figure(figsize=(11, 5.5))
sns.barplot(data=cat, x="Categoria_Corta", y="Valor_por_Par", hue="Flujo_comercial", palette=["#2ca02c", "#ff7f0e"])
plt.title("Gráfico 3: Comparación de Valor Comercial Ponderado por Par (USD/Par)", fontsize=14, fontweight="bold", pad=12)
plt.xlabel("Categoría de Calzado", labelpad=8)
plt.ylabel("Valor Comercial Ponderado (USD / Par)", labelpad=8)
plt.legend(title="Flujo Comercial", frameon=True)
plt.tight_layout()
plt.show()

## 5. Síntesis Ejecutiva (Defensa del Reto)

### Tres Hallazgos Clave:
1. **Déficit Estructural y Crecimiento de Importaciones:** Las importaciones ($18,506 M USD) superan a las exportaciones ($10,917 M USD) en un 69.5% en valor y un 82.8% en pares, con saltos críticos en 2022 y 2024.
2. **Divergencia Valor vs. Cantidad:** El calzado deportivo y sintético acapara el 98.7% del volumen importado compitiendo a precios bajos ($23-$26 USD/par), mientras que el calzado con suela de cuero alcanza más de $45 USD/par pero con apenas el 1% de los pares.
3. **Concentración de Socios:** Grecia, Países Bajos y Vietnam concentran los mayores volúmenes de importación, confirmando la presión de centros logísticos y fábricas asiáticas.

### Dos Limitaciones:
1. **Valor Comercial Registrado vs. Precio Minorista:** El valor aduanero por par es registrado en el dataset de comercio exterior; no incluye márgenes del retail, impuestos ni fletes internos (no equivale al precio en tienda).
2. **Carencia de Datos Internos:** El dataset es de referencia de mercado; para diagnosticar a Manaco con precisión se debe cruzar con su ERP de costos de fábrica y POS de tiendas.

### Pregunta Estratégica para Manaco:
> *"¿Debe Manaco reenfocar su producción fabril exclusivamente hacia el segmento de cuero de alta gama (donde el mercado convalida valores de más de $40-$45 USD/par), dejando la línea deportiva de bajo margen para maquila o distribución externa?"*